## Load raw data and transform

In [ ]:
# import polars as pl
# import time
# import glob

# print("Initializing Polars Lazy Pipeline for Stage 2 (Ranking / MTL)...")
# start_time = time.time()

# root = "/kaggle/input/datasets/nguyenngocanhle/kuairand-1k-parquet/kaggle/working/kuairand_parquet"

# def get_clean_parquet_list(subfolder_pattern):
#     search_path = f"{root}/{subfolder_pattern}"
#     all_files = glob.glob(search_path, recursive=True)
#     clean_files = [f for f in all_files if not f.endswith('.crc') and not f.endswith('_SUCCESS')]
#     if not clean_files:
#         raise ValueError(f"🚨 No valid parquet files found for pattern: {subfolder_pattern}")
#     return clean_files

# log_files = get_clean_parquet_list("log_standard_*/**/*.parquet")
# user_files = get_clean_parquet_list("user_features_1k/**/*.parquet")
# video_files = get_clean_parquet_list("video_features_basic_1k/**/*.parquet")
# category_files = get_clean_parquet_list("kuairand_video_categories/**/*.parquet")
# stat_files = get_clean_parquet_list("video_features_statistic_1k/**/*.parquet")
# embedding_file = ["/kaggle/input/datasets/nguyenngocanhle/video-bge-embeddings/kaggle/working/video_bge_embeddings.parquet"]

# # =========================================================================
# # TABLE 1: Interactions & MTL Targets Table
# # PRUNED: Removed `is_profile_enter` (post-impression behavior / target leakage)
# # =========================================================================
# print("\n[1/3] Generating dense interactions.parquet for MTL Ranking...")
# (
#     pl.scan_parquet(log_files)
#     .select([
#         pl.col("user_id"),
#         pl.col("video_id").alias("target_video_id"),
#         pl.col("time_ms"),
#         pl.col("tab"),
#         # Multi-Task Learning Ground Truth Targets
#         pl.col("is_click"),
#         pl.col("is_like"),
#         pl.col("is_comment"),
#         pl.col("is_forward"),
#         pl.col("is_hate"),
#         pl.col("long_view"),
#         pl.col("play_time_ms")
#     ])
#     .sort("time_ms")
#     .collect(streaming=True)
#     .write_parquet('/kaggle/working/ranking_interactions.parquet')
# )

# # =========================================================================
# # TABLE 2: User Table 
# # PRUNED: Dropped redundant string ranges ('follow_user_num_range', etc.) 
# # in favor of raw continuous metrics which MLPs handle much better.
# # =========================================================================
# print("\n[2/3] Generating user_table.parquet for deep interactions...")

# seq_lazy = (
#     pl.scan_parquet(log_files)
#     .filter(pl.col("is_click") == 1) 
#     .sort(["user_id", "time_ms"])
#     .group_by("user_id")
#     .agg(pl.col("video_id").tail(50).alias("history_sequence")) # Reduced from 100 to 50 for memory/speed
# )

# user_feats = [
#     pl.col("user_active_degree").first(),
#     pl.col("is_live_streamer").first(),
#     pl.col("is_video_author").first(),
#     # Numerical features (log-transformed to stabilize gradients)
#     pl.col("follow_user_num").cast(pl.Float32).log1p().first().alias("follow_user_num_log"),
#     pl.col("fans_user_num").cast(pl.Float32).log1p().first().alias("fans_user_num_log"),
#     pl.col("register_days").cast(pl.Float32).log1p().first().alias("register_days_log"),
# ] + [pl.col(f"onehot_feat{i}").first() for i in range(18)]

# users_lazy = (
#     pl.scan_parquet(user_files)
#     .group_by("user_id")
#     .agg(user_feats)
# )

# (
#     users_lazy
#     .join(seq_lazy, on="user_id", how="left")
#     .with_columns(pl.col("history_sequence").fill_null([])) 
#     .collect(streaming=True)
#     .write_parquet('/kaggle/working/ranking_user_table.parquet')
# )

# # =========================================================================
# # TABLE 3: Item Table
# # PRUNED: Dropped 3rd/4th level categories, category probabilities, 
# # music_type, visible_status, server_width, server_height (irrelevant tech specs).
# # =========================================================================
# print("\n[3/3] Generating item_table.parquet with Dynamic Stats & Metadata...")

# videos_lazy = (
#     pl.scan_parquet(video_files)
#     .with_columns(
#         pl.col("video_duration").cast(pl.Float32).log1p().alias("duration_lognorm")
#     )
#     .group_by("video_id")
#     .agg([
#         pl.col("author_id").first(),
#         pl.col("tag").first(),
#         pl.col("duration_lognorm").first(),
#         pl.col("music_id").first(),
#     ])
# )

# cats_lazy = (
#     pl.scan_parquet(category_files)
#     .group_by("final_video_id")
#     .agg([
#         pl.col("first_level_category_id").first(),
#         pl.col("second_level_category_id").first(),
#     ])
# )

# # Dynamic Statistics (log1p transformed for stable training)
# stats_lazy = (
#     pl.scan_parquet(stat_files)
#     .group_by("video_id")
#     .agg([
#         pl.col("show_cnt").cast(pl.Float32).log1p().first().alias("show_cnt_log"),
#         pl.col("play_progress").cast(pl.Float32).first().alias("avg_play_progress"),
#         pl.col("like_cnt").cast(pl.Float32).log1p().first().alias("like_cnt_log"),
#         pl.col("comment_user_num").cast(pl.Float32).log1p().first().alias("comment_user_num_log"),
#         pl.col("report_cnt").cast(pl.Float32).log1p().first().alias("report_cnt_log")
#     ])
# )

# embeds_lazy = pl.scan_parquet(embedding_file)

# (
#     videos_lazy
#     .join(cats_lazy, left_on="video_id", right_on="final_video_id", how="left")
#     .join(stats_lazy, on="video_id", how="left")
#     .join(embeds_lazy, on="video_id", how="left")
#     .collect(streaming=True)
#     .write_parquet('/kaggle/working/ranking_item_table.parquet')
# )

# print(f"\n✅ Polars Stage 2 Pipeline Complete in {time.time() - start_time:.2f} seconds!")

In [ ]:
!apt-get update && apt-get install -y zip

In [ ]:
import os
import subprocess
from IPython.display import FileLink, display

def download_file(path, download_file_name):
    os.chdir('/kaggle/working/')
    zip_name = f"/kaggle/working/{download_file_name}.zip"
    command = f"zip {zip_name} {path} -r"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("Unable to run zip command!")
        print(result.stderr)
        return
    display(FileLink(f'{download_file_name}.zip'))

# download_file("/kaggle/working", "ranking_item_table.parquet")

## Model architecture

In [ ]:
!pip install polars

In [ ]:
import polars as pl

df_interactions = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/mtl-data/ranking_interactions.parquet")
df_users = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/mtl-data/ranking_user_table.parquet")
df_items = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/mtl-data/ranking_item_table.parquet")

print("==========================================")
print("Interactions")
print("==========================================")
print(df_interactions)

print("\n==========================================")
print("User features")
print("==========================================")
print(df_users)

print("\n==========================================")
print("Item features")
print("==========================================")
print(df_items)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import polars as pl
import numpy as np
import os

# ==========================================
# 1. ARCHITECTURE COMPONENTS (DIN + MMoE)
# ==========================================

# Figures out which past videos the user watched are actually relevant to the current video they are being recommended
class TargetAttention(nn.Module):
    """Deep Interest Network (DIN) Target Attention Module"""
    def __init__(self, embed_dim=32):
        super().__init__()
        self.attn_mlp = nn.Sequential(
            nn.Linear(embed_dim * 4, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, target_item_emb, history_seq_embs):
        # target_item_emb: (Batch, Dim)
        # history_seq_embs: (Batch, SeqLen, Dim)
        seq_len = history_seq_embs.size(1)
        target_expanded = target_item_emb.unsqueeze(1).expand(-1, seq_len, -1)
        
        # Combine [Target, Sequence Item, Target - Sequence Item, Target * Sequence Item]
        concat_feat = torch.cat([
            target_expanded,
            history_seq_embs,
            target_expanded - history_seq_embs,
            target_expanded * history_seq_embs
        ], dim=-1)
        
        attn_weights = self.attn_mlp(concat_feat) # (Batch, SeqLen, 1)
        attn_weights = F.softmax(attn_weights, dim=1)
        
        # Weighted sum of past user history vectors
        user_interest_vector = torch.sum(attn_weights * history_seq_embs, dim=1)
        return user_interest_vector


class MMoELayer(nn.Module):
    """Multi-gate Mixture-of-Experts (MMoE) Layer"""
    def __init__(self, input_dim, num_experts=4, expert_hidden_dim=128, num_tasks=7):
        super().__init__()
        self.num_experts = num_experts
        self.num_tasks = num_tasks
        
        # 2. Two-Layer Experts: Added a second linear transformation and activation.
        # This increases the capacity of each expert to model nonlinear feature relationships.
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, expert_hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(expert_hidden_dim, expert_hidden_dim), # Second Layer
                nn.ReLU()
            ) for _ in range(num_experts)
        ])
        
        # Task Gating Networks
        self.gates = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, num_experts),
                nn.Softmax(dim=-1)
            ) for _ in range(num_tasks)
        ])

    def forward(self, x):
        # x shape: (Batch, input_dim)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1) # (Batch, NumExperts, HiddenDim)
        
        task_inputs = []
        for gate in self.gates:
            gate_weights = gate(x).unsqueeze(-1) # (Batch, NumExperts, 1)
            task_representation = torch.sum(gate_weights * expert_outputs, dim=1) # (Batch, HiddenDim)
            task_inputs.append(task_representation)
            
        return task_inputs


class MMoERankingModel(nn.Module):
    def __init__(self, num_items, num_categories, num_tags, item_embed_dim=32, nlp_embed_dim=512):
        super().__init__()
        
        # Embeddings
        self.item_embedding = nn.Embedding(num_items, item_embed_dim, padding_idx=0)
        self.cat1_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.cat2_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.tag_embedding  = nn.Embedding(num_tags, 8, padding_idx=0)
        
        self.target_attention = TargetAttention(embed_dim=item_embed_dim)
        
        # NLP Reduction MLP
        self.text_mlp = nn.Sequential(nn.Linear(nlp_embed_dim, 32), nn.ReLU())
        
        # Original Input Dimension: 150
        raw_input_dim = 32 + 24 + 32 + 8 + 8 + 8 + 1 + 5 + 32
        
        # 1. Feature Preprocessing MLP:
        # We pass the raw concatenated features through this MLP before hitting the MMoE.
        # This lets distinct features interact globally before being routed to experts.
        mmoe_input_dim = 128
        self.feature_preprocessing = nn.Sequential(
            nn.Linear(raw_input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, mmoe_input_dim),
            nn.ReLU()
        )
        
        # MMoE Engine (Now receives the 128-dim preprocessed features)
        self.mmoe = MMoELayer(input_dim=mmoe_input_dim, num_experts=4, expert_hidden_dim=64, num_tasks=7)
        
        # 3. Two-Layer Task Towers:
        # A helper function builds deeper task-specific heads instead of single linear layers.
        def build_task_tower(hidden_dim):
            return nn.Sequential(
                nn.Linear(hidden_dim, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            )

        # 7 Multi-Task Prediction Heads
        self.head_click = build_task_tower(64)       # Task 1: pCTR
        self.head_like = build_task_tower(64)        # Task 2: pLike
        self.head_comment = build_task_tower(64)     # Task 3: pComment
        self.head_forward = build_task_tower(64)     # Task 4: pForward
        self.head_hate = build_task_tower(64)        # Task 5: pHate
        self.head_long_view = build_task_tower(64)   # Task 6: pFinish / pLongView
        self.head_watch_time = build_task_tower(64)  # Task 7: Expected Watch Time

    def forward(self, history_seq, user_static, item_id, cat1_id, cat2_id, tags, item_stats, dur_log, nlp_vec):
        # Item Embeddings
        target_item_emb = self.item_embedding(item_id)
        history_seq_embs = self.item_embedding(history_seq)
        
        # Target Attention Over History
        user_interest = self.target_attention(target_item_emb, history_seq_embs)
        
        c1_emb = self.cat1_embedding(cat1_id)
        c2_emb = self.cat2_embedding(cat2_id)
        tag_emb = self.tag_embedding(tags).sum(dim=1)
        text_feat = self.text_mlp(nlp_vec)
        dur_feat = dur_log.unsqueeze(1)
        
        # Concatenate ALL User, Context, and Item features into a Single Representation
        dense_concat = torch.cat([
            user_interest, user_static, target_item_emb, 
            c1_emb, c2_emb, tag_emb, dur_feat, item_stats, text_feat
        ], dim=-1)
        
        # 1. Pass through Feature Preprocessing MLP
        preprocessed_features = self.feature_preprocessing(dense_concat)
        
        # MMoE Forward Pass
        task_representations = self.mmoe(preprocessed_features)
        
        # 4. Remove Sigmoids: BCEWithLogitsLoss requires raw logits!
        # 5. Remove F.relu on watch_time: We are predicting log-transformed watch time, 
        #    which can theoretically be modeled as a continuous real number.
        outputs = {
            "p_click_logits": self.head_click(task_representations[0]),
            "p_like_logits": self.head_like(task_representations[1]),
            "p_comment_logits": self.head_comment(task_representations[2]),
            "p_forward_logits": self.head_forward(task_representations[3]),
            "p_hate_logits": self.head_hate(task_representations[4]),
            "p_long_view_logits": self.head_long_view(task_representations[5]),
            "watch_time_log": self.head_watch_time(task_representations[6]) 
        }
        return outputs


# ==========================================
# 2. MULTI-TASK LOSS FUNCTION
# ==========================================

class MultiTaskLoss(nn.Module):
    # Notice the watch_time weight is adjusted up slightly since log(ms) shrinks the loss scale.
    def __init__(self, weights={"click": 1.0, "like": 2.0, "comment": 2.0, "forward": 3.0, "hate": 3.0, "long_view": 1.5, "watch_time_log": 0.5}):
        super().__init__()
        # 4. Replace BCELoss with BCEWithLogitsLoss for far better numerical stability
        self.bce_logits = nn.BCEWithLogitsLoss()
        self.mse = nn.MSELoss()
        self.w = weights

    def forward(self, preds, targets):
        # Calculate BCE loss using raw un-sigmoid'd logits
        l_click = self.bce_logits(preds["p_click_logits"].squeeze(), targets["is_click"].float())
        l_like = self.bce_logits(preds["p_like_logits"].squeeze(), targets["is_like"].float())
        l_comment = self.bce_logits(preds["p_comment_logits"].squeeze(), targets["is_comment"].float())
        l_forward = self.bce_logits(preds["p_forward_logits"].squeeze(), targets["is_forward"].float())
        l_hate = self.bce_logits(preds["p_hate_logits"].squeeze(), targets["is_hate"].float())
        l_long_view = self.bce_logits(preds["p_long_view_logits"].squeeze(), targets["long_view"].float())
        
        # 5. Predict Log-Transformed Watch Time. 
        # We use torch.log1p (log(1 + x)) on the targets to safely avoid log(0)
        target_watch_time_log = torch.log1p(targets["play_time_ms"].float())
        l_watch_time = self.mse(preds["watch_time_log"].squeeze(), target_watch_time_log)
        
        total_loss = (
            self.w["click"] * l_click +
            self.w["like"] * l_like +
            self.w["comment"] * l_comment +
            self.w["forward"] * l_forward +
            self.w["hate"] * l_hate +
            self.w["long_view"] * l_long_view +
            self.w["watch_time_log"] * l_watch_time
        )
        return total_loss

## Data Transforming and Model Traing

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, mean_absolute_error
import polars as pl
import numpy as np
import os

# Force XLA to target the actual TPU hardware
os.environ['PJRT_DEVICE'] = 'TPU' 
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as xla_pl

# ==========================================
# 1. MULTI-TASK DATASET (FEATURES + 7 TARGETS)
# ==========================================

class MMoEDataset(Dataset):
    def __init__(self, interactions_df, user_df, item_df, num_items, num_tags):
        # 1. Interaction Anchors & Multi-Task Targets
        self.user_ids = interactions_df['user_id'].to_numpy()
        self.target_ids = interactions_df['target_video_id'].to_numpy()
        
        # Extract the 7 Ground Truth Targets
        self.targets = interactions_df.select([
            'is_click', 'is_like', 'is_comment', 'is_forward', 
            'is_hate', 'long_view', 'play_time_ms'
        ]).to_numpy()
        
        valid_item_ids = item_df['video_id'].to_numpy()
        
        # 2. Item Tensors (Including Dynamic Stats for Ranking)
        print("Building Item Tensors for Ranking in RAM...")
        self.item_nlp_tensor = torch.zeros((num_items, 512), dtype=torch.float32)
        vecs = [v if v is not None else [0.0]*512 for v in item_df['nlp_vector'].to_list()]
        self.item_nlp_tensor[valid_item_ids] = torch.tensor(vecs, dtype=torch.float32)

        self.cat1_tensor = torch.zeros(num_items, dtype=torch.long)
        self.cat1_tensor[valid_item_ids] = torch.tensor(np.maximum(0, item_df['first_level_category_id'].fill_null(0).to_numpy()), dtype=torch.long)

        self.cat2_tensor = torch.zeros(num_items, dtype=torch.long)
        self.cat2_tensor[valid_item_ids] = torch.tensor(np.maximum(0, item_df['second_level_category_id'].fill_null(0).to_numpy()), dtype=torch.long)

        self.dur_tensor = torch.zeros(num_items, dtype=torch.float32)
        self.dur_tensor[valid_item_ids] = torch.tensor(item_df['duration_lognorm'].fill_null(0.0).to_numpy(), dtype=torch.float32)
        
        # Item Dynamic Stats (5 Features)
        self.item_stats_tensor = torch.zeros((num_items, 5), dtype=torch.float32)
        stats_cols = ['show_cnt_log', 'avg_play_progress', 'like_cnt_log', 'comment_user_num_log', 'report_cnt_log']
        stats_exprs = [pl.col(c).cast(pl.Float32, strict=False) for c in stats_cols]
        stats_matrix = np.nan_to_num(item_df.select(stats_exprs).fill_null(0.0).to_numpy(), nan=0.0)
        self.item_stats_tensor[valid_item_ids] = torch.tensor(stats_matrix, dtype=torch.float32)

        # Tags Parsing
        MAX_TAGS = 5
        self.tag_tensor = torch.zeros((num_items, MAX_TAGS), dtype=torch.long)
        tag_strings = item_df['tag'].fill_null("").to_list()
        for idx, t_str in zip(valid_item_ids, tag_strings):
            if t_str and t_str != "UNKNOWN":
                t_ints = [int(x) for x in t_str.split(",") if x.isdigit()]
                length = min(len(t_ints), MAX_TAGS)
                self.tag_tensor[idx, :length] = torch.tensor(t_ints[:length], dtype=torch.long)

        # 3. User Tensors
        print("Building User Tensors for Ranking in RAM...")
        max_user_id = user_df['user_id'].max() + 1
        valid_user_ids = user_df['user_id'].to_numpy()
        
        # Grab the 24 static user features prepared in Polars
        static_cols = [c for c in user_df.columns if c not in ['user_id', 'history_sequence']]
        self.user_static_tensor = torch.zeros((max_user_id, len(static_cols)), dtype=torch.float32)
        static_exprs = [pl.col(c).cast(pl.Float32, strict=False) for c in static_cols]
        static_matrix = np.nan_to_num(user_df.select(static_exprs).fill_null(0.0).to_numpy(), nan=0.0)
        
        self.user_static_tensor[valid_user_ids] = torch.tensor(static_matrix, dtype=torch.float32)

        self.user_history_lookup = {
            u: np.array(h, dtype=np.int64) if h is not None else np.array([], dtype=np.int64)
            for u, h in zip(valid_user_ids, user_df['history_sequence'].to_list())
        }

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        u_id = self.user_ids[idx]
        i_id = self.target_ids[idx]
        targs = self.targets[idx]

        # History padding (Max 50 for MMoE to save memory)
        raw_history = self.user_history_lookup.get(u_id, np.array([], dtype=np.int64))
        MAX_SEQ_LEN = 50
        padded_history = np.zeros(MAX_SEQ_LEN, dtype=np.int64)
        if len(raw_history) > 0:
            length = min(len(raw_history), MAX_SEQ_LEN)
            padded_history[-length:] = raw_history[-length:]
            
        return {
            # Features
            "history_seq": torch.tensor(padded_history, dtype=torch.long),
            "user_static": self.user_static_tensor[u_id],
            "item_id": torch.tensor(i_id, dtype=torch.long),
            "cat1_id": self.cat1_tensor[i_id],
            "cat2_id": self.cat2_tensor[i_id],
            "tags": self.tag_tensor[i_id],
            "item_stats": self.item_stats_tensor[i_id],
            "dur_log": self.dur_tensor[i_id],
            "nlp_vec": self.item_nlp_tensor[i_id],
            
            # Multi-Task Targets
            "is_click": torch.tensor(targs[0], dtype=torch.float32),
            "is_like": torch.tensor(targs[1], dtype=torch.float32),
            "is_comment": torch.tensor(targs[2], dtype=torch.float32),
            "is_forward": torch.tensor(targs[3], dtype=torch.float32),
            "is_hate": torch.tensor(targs[4], dtype=torch.float32),
            "long_view": torch.tensor(targs[5], dtype=torch.float32),
            "play_time_ms": torch.tensor(targs[6], dtype=torch.float32),
        }

# ==========================================
# 2. RANKING EVALUATION (AUC & MAE)
# ==========================================

def safe_auc(y_true, y_pred):
    """Calculates AUC safely, avoiding errors if a batch lacks positive samples."""
    if len(np.unique(y_true)) > 1:
        return roc_auc_score(y_true, y_pred)
    return 0.5 # Return random chance if only 1 class exists in the split

def evaluate_mmoe(model, dataloader, device):
    model.eval()
    
    # Storage for calculating overall metrics
    all_preds = {k: [] for k in ["p_click", "p_like", "p_comment", "p_forward", "p_hate", "p_long_view", "watch_time"]}
    all_targs = {k: [] for k in ["is_click", "is_like", "is_comment", "is_forward", "is_hate", "long_view", "play_time_ms"]}
    total_loss = 0.0
    criterion = MultiTaskLoss()
    
    with torch.no_grad():
        for batch in dataloader:
            # 1. Forward Pass
            preds = model(
                batch['history_seq'], batch['user_static'], batch['item_id'], 
                batch['cat1_id'], batch['cat2_id'], batch['tags'], 
                batch['item_stats'], batch['dur_log'], batch['nlp_vec']
            )
            
            # 2. Compute Batch Loss (Loss function expects the raw logits and log-time!)
            loss = criterion(preds, batch)
            total_loss += loss.item()
            
            # 3. Pull predictions back to CPU for Scikit-Learn evaluation
            # FIX: Apply Sigmoid to convert logits back into [0, 1] probabilities for AUC
            all_preds["p_click"].extend(torch.sigmoid(preds["p_click_logits"]).squeeze().cpu().numpy())
            all_preds["p_like"].extend(torch.sigmoid(preds["p_like_logits"]).squeeze().cpu().numpy())
            all_preds["p_comment"].extend(torch.sigmoid(preds["p_comment_logits"]).squeeze().cpu().numpy())
            all_preds["p_forward"].extend(torch.sigmoid(preds["p_forward_logits"]).squeeze().cpu().numpy())
            all_preds["p_hate"].extend(torch.sigmoid(preds["p_hate_logits"]).squeeze().cpu().numpy())
            all_preds["p_long_view"].extend(torch.sigmoid(preds["p_long_view_logits"]).squeeze().cpu().numpy())
            
            # FIX: Apply expm1 (exponential minus 1) to convert log watch time back to milliseconds for MAE
            all_preds["watch_time"].extend(torch.expm1(preds["watch_time_log"]).squeeze().cpu().numpy())
            
            # Ground Truth Targets
            all_targs["is_click"].extend(batch["is_click"].cpu().numpy())
            all_targs["is_like"].extend(batch["is_like"].cpu().numpy())
            all_targs["is_comment"].extend(batch["is_comment"].cpu().numpy())
            all_targs["is_forward"].extend(batch["is_forward"].cpu().numpy())
            all_targs["is_hate"].extend(batch["is_hate"].cpu().numpy())
            all_targs["long_view"].extend(batch["long_view"].cpu().numpy())
            all_targs["play_time_ms"].extend(batch["play_time_ms"].cpu().numpy())

    # Calculate final metrics across the entire validation epoch
    metrics = {
        "loss": total_loss / len(dataloader),
        "auc_click": safe_auc(all_targs["is_click"], all_preds["p_click"]),
        "auc_like": safe_auc(all_targs["is_like"], all_preds["p_like"]),
        "auc_comment": safe_auc(all_targs["is_comment"], all_preds["p_comment"]),
        "auc_forward": safe_auc(all_targs["is_forward"], all_preds["p_forward"]),
        "auc_hate": safe_auc(all_targs["is_hate"], all_preds["p_hate"]),
        "auc_long_view": safe_auc(all_targs["long_view"], all_preds["p_long_view"]),
        "mae_watch_time": mean_absolute_error(all_targs["play_time_ms"], all_preds["watch_time"])
    }
    return metrics


# ==========================================
# 3. TPU TRAINING LOOP FOR MMoE
# ==========================================

def train_mmoe_tpu(df_interactions, df_users, df_items, num_items, num_categories, num_tags):
    # Time-based split to prevent target leakage
    df_interactions = df_interactions.sort("time_ms")
    split_idx = int(len(df_interactions) * 0.8)
    train_df = df_interactions[:split_idx]
    val_df = df_interactions[split_idx:]
    
    train_dataset = MMoEDataset(train_df, df_users, df_items, num_items, num_tags)
    val_dataset = MMoEDataset(val_df, df_users, df_items, num_items, num_tags)
    
    device = xm.xla_device()
    print(f"🚀 Initializing MMoE Training on device: {device}")

    # DataLoaders wrapped in XLA Parallel Loader for TPU bottleneck prevention
    train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, num_workers=4, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=2048, shuffle=False, num_workers=4, drop_last=False)
    
    train_device_loader = xla_pl.MpDeviceLoader(train_loader, device)
    val_device_loader = xla_pl.MpDeviceLoader(val_loader, device)
    
    # Instantiate the MMoE Model (Assumes MMoERankingModel & MultiTaskLoss are defined)
    model = MMoERankingModel(
        num_items=num_items, 
        num_categories=num_categories, 
        num_tags=num_tags
    ).to(device)
    
    criterion = MultiTaskLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    
    epochs = 10
    best_auc_like = 0.0
    patience_counter = 0
    patience = 2
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        # Batch Loop
        for batch_idx, batch in enumerate(train_device_loader):
            optimizer.zero_grad()
            
            # Forward Pass
            preds = model(
                batch['history_seq'], batch['user_static'], batch['item_id'], 
                batch['cat1_id'], batch['cat2_id'], batch['tags'], 
                batch['item_stats'], batch['dur_log'], batch['nlp_vec']
            )
            
            # Combined Loss
            loss = criterion(preds, batch)
            loss.backward()

            # Gradient Clipping is vital for MTL stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            
            # XLA Optimizer Step (Compiles Graph)
            xm.optimizer_step(optimizer, barrier=True)
            train_loss += loss.item()
            
            # if batch_idx % 100 == 0:
            #     print(f"Epoch {epoch+1} | Batch {batch_idx} | Current Loss: {loss.item():.4f}")

        # Evaluation Phase
        print(f"Calculating Validation Metrics for Epoch {epoch+1}...")
        val_metrics = evaluate_mmoe(model, val_device_loader, device)
        
        print(f"\n--- Epoch {epoch+1} Complete ---")
        print(f"📉 Train Loss: {train_loss/len(train_device_loader):.4f} | Val Loss: {val_metrics['loss']:.4f}")
        print(f"🎯 AUC Clicks : {val_metrics['auc_click']:.4f}")
        print(f"❤️ AUC Likes  : {val_metrics['auc_like']:.4f}")
        print(f"💬 AUC Comment: {val_metrics['auc_comment']:.4f}")
        print(f"⏩ AUC Forward: {val_metrics['auc_forward']:.4f}")
        print(f"😡 AUC Hate   : {val_metrics['auc_hate']:.4f}")
        print(f"📺 AUC L-View : {val_metrics['auc_long_view']:.4f}")
        print(f"⏱️ MAE Watch  : {val_metrics['mae_watch_time']:.1f} ms")
        
        # --- EARLY STOPPING LOGIC ---
        # Save based on a high-intent behavior (Like AUC) rather than just Clicks
        if val_metrics["auc_like"] > best_auc_like:
            best_auc_like = val_metrics["auc_like"]
            xm.save(model.state_dict(), "best_mmoe_ranking_tpu.pth")
            print(f"🔥 New best model saved! (AUC Like: {best_auc_like:.4f})\n")
            patience_counter = 0 # Reset counter on improvement
        else:
            patience_counter += 1
            print(f"⚠️ No improvement in AUC Like. Patience: {patience_counter}/{patience}\n")
            
            if patience_counter >= patience:
                print(f"🛑 Early stopping triggered after {epoch+1} epochs! Best AUC Like was {best_auc_like:.4f}.")
                break

    return model, device

#-----------------------------------------------------------
#                   EXECUTION
#-----------------------------------------------------------
# 1. Calculate num_items
max_item_id = df_items["video_id"].max() or 0
max_int_id = df_interactions["target_video_id"].max() or 0
max_hist_id = df_users.select(pl.col("history_sequence").list.explode().max()).to_series()[0] or 0
num_items = int(max(max_item_id, max_int_id, max_hist_id)) + 1

# 2. Calculate num_categories
max_cat1 = df_items['first_level_category_id'].max() or 0
max_cat2 = df_items['second_level_category_id'].max() or 0
num_categories = int(max(max_cat1, max_cat2)) + 1

# 3. Calculate num_tags globally before dataset creation
tag_strings = df_items['tag'].fill_null("").to_list()
global_max_tag = 0
for t_str in tag_strings:
    if t_str and t_str != "UNKNOWN":
        t_ints = [int(x) for x in t_str.split(",") if x.isdigit()]
        if t_ints:
            global_max_tag = max(global_max_tag, max(t_ints))
num_tags = global_max_tag + 1

print(f"📊 Dataset Boundaries -> Items: {num_items:,} | Categories: {num_categories:,} | Tags: {num_tags:,}")

train_mmoe_tpu(df_interactions, df_users, df_items, num_items, num_categories, num_tags)

In [ ]:
download_file("/kaggle/working/best_mmoe_ranking_tpu.pth", "best_mmoe_ranking_tpu.pth")